In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpga3qi4dt".


In [2]:
%%cuda
#include <iostream>

// THE KERNEL
__global__ void convolve1D(float* in, float* out, int n) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;

    if (i < n) {
        float sum = 0.0f;
        int count = 0;

        // Iterate through neighbors (i-1, i, i+1)
        for (int mask = -1; mask <= 1; mask++) {
            int index = i + mask;

            // Boundary Check: Don't read outside the array
            if (index >= 0 && index < n) {
                sum += in[index];
                count++;
            }
        }
        out[i] = sum / count;
    }
}

int main() {
    int n = 10;
    size_t size = n * sizeof(float);

    float h_in[] = {0, 10, 20, 10, 0, 5, 10, 5, 0, 0};
    float h_out[10];

    float *d_in, *d_out;
    cudaMalloc(&d_in, size);
    cudaMalloc(&d_out, size);

    cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice);

    convolve1D<<<1, 256>>>(d_in, d_out, n);

    cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost);

    std::cout << "In:  ";
    for(int i=0; i<n; i++) printf("%.1f ", h_in[i]);
    std::cout << "\nOut: ";
    for(int i=0; i<n; i++) printf("%.1f ", h_out[i]);
    std::cout << std::endl;

    cudaFree(d_in); cudaFree(d_out);
    return 0;
}

In:  0.0 10.0 20.0 10.0 0.0 5.0 10.0 5.0 0.0 0.0 
Out: 5.0 10.0 13.3 10.0 5.0 5.0 6.7 5.0 1.7 0.0 



Optmized version

In [3]:
%%cuda
#include <iostream>

#define RADIUS 1
#define BLOCK_SIZE 256

__global__ void convolve1DShared(float* in, float* out, int n) {
    // Shared memory includes the main data + halos on both sides
    __shared__ float temp[BLOCK_SIZE + 2 * RADIUS];

    int g_idx = blockIdx.x * blockDim.x + threadIdx.x; // Global index
    int l_idx = threadIdx.x + RADIUS;                 // Local index in shared memory

    // 1. Load main data into shared memory
    if (g_idx < n) {
        temp[l_idx] = in[g_idx];
    } else {
        temp[l_idx] = 0.0f;
    }

    // 2. Load Halos (Left and Right)
    if (threadIdx.x < RADIUS) {
        // Left Halo
        temp[l_idx - RADIUS] = (g_idx - RADIUS >= 0) ? in[g_idx - RADIUS] : 0.0f;
        // Right Halo
        int right_idx = g_idx + BLOCK_SIZE;
        temp[l_idx + BLOCK_SIZE] = (right_idx < n) ? in[right_idx] : 0.0f;
    }

    // 3. WAIT for all threads to finish loading
    __syncthreads();

    // 4. Compute convolution using Shared Memory
    if (g_idx < n) {
        float sum = 0.0f;
        for (int i = -RADIUS; i <= RADIUS; i++) {
            sum += temp[l_idx + i];
        }
        out[g_idx] = sum / 3.0f;
    }
}

int main() {
    int n = 10;
    size_t size = n * sizeof(float);
    float h_in[] = {0, 10, 20, 10, 0, 5, 10, 5, 0, 0};
    float h_out[10];

    float *d_in, *d_out;
    cudaMalloc(&d_in, size);
    cudaMalloc(&d_out, size);
    cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice);

    convolve1DShared<<<1, BLOCK_SIZE>>>(d_in, d_out, n);

    cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost);

    printf("Optimized Output: ");
    for(int i=0; i<n; i++) printf("%.1f ", h_out[i]);
    printf("\n");

    cudaFree(d_in); cudaFree(d_out);
    return 0;
}

Optimized Output: 3.3 10.0 13.3 10.0 5.0 5.0 6.7 5.0 1.7 0.0 

